# 🛢️ PumpGuard AI — Workshop IoT Predictive Maintenance
**v2.0** &nbsp;·&nbsp; Duration: 3–4 hours &nbsp;·&nbsp; Level: Intermediate &nbsp;·&nbsp; Platform: Google Colab

---

## 🎯 Objectives

Build a real-world IoT Predictive Maintenance system step by step, covering:

| Layer | Technology | Role |
|-------|------------|------|
| Sensor | Python simulator | Simulates 52 industrial sensors |
| Transport | Mosquitto MQTT | Local broker, latency ~1ms |
| Pipeline | Node-RED | Trend analysis & anomaly detection |
| Backend | FastAPI + WebSocket | API server, real-time broadcast |
| AI | Groq (llama-3.3-70b) | Analysis & maintenance recommendations |
| Alerting | Resend | Automated email alerts |
| Dashboard | HTML / JS / Chart.js | Real-time browser interface |

---

## 🏗️ Architecture

```
[mqtt_replay.py] → [Mosquitto :1883] → [Node-RED :1880]
                                               │
                        ┌──────────────────────┤
                        │                      │
               POST /simulate/inject    POST /analyze
               (enriched sensor_update) (on anomaly)
                        │                      │
                   [FastAPI :8000 on Colab] ───┘
                        │ WebSocket
                   [Dashboard – Browser]
                        │ ngrok tunnel
                   [Public URL]
```

---

## 📋 How to use this notebook

> Each part = **theory → practice → confirmation**.
> Source files are provided by the instructor — you will **upload each file** to Colab as instructed.


---
# ⚙️ Pre-Workshop Setup (Prerequisites)

Complete the following before starting the workshop:

---

## 1 · Google Colab

Sign in to **[colab.research.google.com](https://colab.research.google.com/)** with your Google account.
Select **Runtime → Change runtime type → T4 GPU** (CPU is also fine).

---

## 2 · Resend API key *(for email alerts)*

1. Go to **[resend.com](https://resend.com/)** → Sign up for free
2. Go to **API Keys** → create a new key → copy it
3. Save the following:
```
RESEND_KEY = re_xxxxxxxxxxxx
ALERT_EMAIL = your@email.com
```

---

## 3 · Groq API key *(for AI analysis)*

1. Go to **[aistudio.google.com/apikey](https://console.groq.com)** → create a key
2. Save the following:
```
GROQ_API_KEY = gsk_xxxxxxxxxxxxxxxxxxxx
```

---

## 4 · Prepare files (download from GitHub repo)

| File | Description |
|------|-------------|
| `server.py` | FastAPI backend |
| `index.html` | Main dashboard |
| `control.html` | Control panel |
| `flows.json` | Node-RED pipeline config |
| `sensor.csv` | Pump Sensor dataset (Kaggle) |
| `analyze_sensors.py` | Analyse CSV → generate config JSON |
| `mqtt_replay.py` | Replay sensor data over MQTT |


---
# Part 1 — Setup & Workspace Creation

## 1.1 Check Runtime


In [ ]:
import sys, platform, subprocess
print(f'Python  : {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
mem = subprocess.run(['free','-h'], capture_output=True, text=True).stdout
print('Memory:\n' + mem)
assert sys.version_info >= (3,8), 'Python 3.8+ required'
print('✅ Runtime OK')


## 1.2 Create Project Directory Structure

Run this cell to create the folder structure.
You will upload files here as directed in each part of the notebook.

```
/content/pump-iot-demo/
├── backend/        ← server.py, .env          (Part 3)
├── dashboard/      ← index.html, control.html (Part 6)
├── nodered/        ← flows.json               (Part 5)
└── scripts/        ← mqtt_replay.py           (Part 7)
```


In [ ]:
import os
PROJ = '/content/pump-iot-demo'
for d in ['backend', 'dashboard', 'nodered', 'scripts']:
    os.makedirs(os.path.join(PROJ, d), exist_ok=True)
    print(f'  📁 {PROJ}/{d}/')
print('\n✅ Directory structure created. Ready to receive files!')


---
# Part 2 — Install Dependencies

## 2.1 Python packages

| Package | Role |
|---------|------|
| `fastapi` + `uvicorn` | Web framework + ASGI server |
| `paho-mqtt` | MQTT client for subscribing from the broker |
| `python-dotenv` | Read variables from `.env` file |
| `groq` | Groq SDK |
| `resend` | Email API |
| `pyngrok` | Create public ngrok URL |



### 📋 Practice: Upload `requirements.txt`

*Package list — must be present before running pip install.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/backend/`**
3. Drag and drop **`requirements.txt`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/backend/requirements.txt` |
| 📍 Upload to | `/content/pump-iot-demo/backend/requirements.txt` |

> After uploading, run the cell below to install all packages at once.


In [ ]:
import os, subprocess, sys

_req = '/content/pump-iot-demo/backend/requirements.txt'
assert os.path.exists(_req), '❌ requirements.txt not found — upload it following the instructions above'
print(f'✅ requirements.txt found')

print('📥 Installing all packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', _req],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠️  Error:')
    print(result.stderr[-1000:])
else:
    print('✅ All packages installed!')


## 2.2 Install Mosquitto & Node-RED

This workshop uses **Mosquitto** (local MQTT broker) and **Node-RED** (running on Colab, exposed via tunnel).
Run this cell once before getting started:


In [ ]:
print('📥 Installing Mosquitto...')
!apt-get install -y -q mosquitto mosquitto-clients 2>&1 | tail -3
print('📥 Installing Node.js 20 + Node-RED...')
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>&1 | tail -3
!apt-get install -y -q nodejs 2>&1 | tail -3
!npm install -g --quiet node-red 2>&1 | tail -3
print('✅ Mosquitto + Node-RED installed successfully')


---
# Part 3 — Backend: `server.py`

## What does server.py do?

This file is the processing hub of the system:

```
           ┌──────────────────────────────────────────────┐
           │                server.py                    │
           │                                             │
MQTT ──►  MQTTBridge ──────────────────────────────────► │
           │  (raw fallback broadcast while              │
           │   Node-RED buffer fills up)                 │
           │                                             │
REST ──►  /simulate/inject ──► WebSocket ──► Dashboard  │
           │  (enriched data from Node-RED)              │
           │                                             │
REST ──►  /analyze ──► Groq AI ──► WebSocket ──► Dashboard│
           │  (called by Node-RED on anomaly)            │
           │                                             │
REST ──►  /alert ──► Email (Resend)                     │
           │  (called by dashboard on WARNING/CRITICAL)  │
           └──────────────────────────────────────────────┘
```

**Key endpoints:**

| Endpoint | Method | Called by | Purpose |
|----------|--------|-----------|--------|
| `/ws` | WebSocket | Dashboard | Real-time sensor stream |
| `/simulate/inject` | POST | Node-RED | Broadcast enriched sensor data (with trends) |
| `/analyze` | POST | Node-RED | Trigger AI analysis on anomaly |
| `/alert` | POST | Dashboard | Send email alert |
| `/control/{state}` | POST | Control panel | Simulate Normal/Warning/Critical |
| `/health` | GET | — | Status check |


### 📋 Practice: Upload `server.py`

*Main backend file — required before starting FastAPI.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/backend/`**
3. Drag and drop **`server.py`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/backend/server.py` |
| 📍 Upload to | `/content/pump-iot-demo/backend/server.py` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/backend/server.py'
assert os.path.exists(_p), '❌ server.py not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ server.py ({_kb} KB) — OK')


### 📋 Practice: Upload `email_alert.html`

*Email template file — used by the backend to render alert emails.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/backend/`**
3. Drag and drop **`email_alert.html`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/backend/email_alert.html` |
| 📍 Upload to | `/content/pump-iot-demo/backend/email_alert.html` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/backend/email_alert.html'
assert os.path.exists(_p), '❌ email_alert.html not found — upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ email_alert.html ({_kb} KB) — OK')


## 3.2 Create the `.env` file

The `.env` file stores API keys — **created via code** since it contains personal credentials. **Never** commit this file to Git.

**Required variables:**

| Variable | Description |
|----------|-------------|
| `MQTT_HOST` | `localhost` — Mosquitto runs on Colab |
| `GROQ_API_KEY` | Groq API key from console.groq.com |
| `RESEND_API_KEY` | (Optional) Email alerts via resend.com |
| `ALERT_TO` | Recipient email for alerts |



In [ ]:
import os
# ════════════════════════════════════════
# ✏️ Fill in your details below
# ════════════════════════════════════════
GROQ_KEY = ''   # get yours at: https://console.groq.com

RESEND_KEY = ''
ALERT_FROM = 'PumpGuard AI <onboarding@resend.dev>'
ALERT_TO   = 'your@email.com'
# ════════════════════════════════════════

assert GROQ_KEY.strip(), '❌ Fill in GROQ_KEY above before running'

env_lines = [
    '# PumpGuard AI — Environment Config',
    'MQTT_HOST=localhost',
    'MQTT_PORT=1883',
    f'GROQ_API_KEY={GROQ_KEY}',
    f'RESEND_API_KEY={RESEND_KEY}',
    f'ALERT_FROM={ALERT_FROM}',
    f'ALERT_TO={ALERT_TO}',
]
env_path = '/content/pump-iot-demo/backend/.env'
with open(env_path, 'w') as f:
    f.write('\n'.join(env_lines))
print('✅ .env created: ' + env_path)
print()
print('Contents (values masked):')
with open(env_path) as f:
    for line in f:
        if '=' in line and not line.startswith('#'):
            k, v = line.strip().split('=', 1)
            masked = (v[:4]+'***') if len(v) > 5 else ('(empty)' if not v else v)
            print(f'  {k:25s} = {masked}')


---
# Part 4 — MQTT Broker

## What is MQTT?

**MQTT** is a lightweight pub/sub protocol — the industry standard for IoT.

```
[mqtt_replay.py]  ──publish──►  [MQTT Broker]  ──subscribe──►  [FastAPI → WebSocket → Dashboard]
                                  topic: pump/sensors
```

Workshop uses **Mosquitto** running directly on Colab — no cloud broker needed, **lower latency** (~1ms vs ~50ms for cloud MQTT).

---
## ☁️ Start Mosquitto & verify connection


---
## 4.1 Start Mosquitto (local MQTT broker)


In [ ]:
import subprocess, time, os

# Stop any existing mosquitto process
subprocess.run(['pkill', '-f', 'mosquitto'], capture_output=True)
time.sleep(0.5)

# Minimal config at /tmp — NO pid_file entry.
# When run with -c /tmp/mosquitto.conf, mosquitto only reads this file,
# ignoring /etc/mosquitto/mosquitto.conf (avoids pid dir permission errors).
with open('/tmp/mosquitto.conf', 'w') as f:
    f.write('listener 1883\nallow_anonymous true\n')

# Use Popen because mosquitto is a daemon — must keep running in the background
mosq_proc = subprocess.Popen(
    ['mosquitto', '-c', '/tmp/mosquitto.conf'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
time.sleep(1)

if mosq_proc.poll() is None:
    print(f'✅ Mosquitto running (PID {mosq_proc.pid}) — port 1883')
    r = subprocess.run(
        ['mosquitto_pub', '-h', 'localhost', '-t', 'test', '-m', 'ok'],
        capture_output=True, timeout=3
    )
    print('   Publish test:', 'OK ✅' if r.returncode == 0 else r.stderr.decode())
else:
    err = mosq_proc.stderr.read().decode(errors='replace')
    print('❌ Mosquitto failed:\n' + err)


In [ ]:
# Verify paho-mqtt can connect to Mosquitto
import paho.mqtt.client as mqtt, os, threading
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

HOST = os.environ.get('MQTT_HOST', 'localhost')
PORT = int(os.environ.get('MQTT_PORT', 1883))

_ok = threading.Event()
def _on_connect(c, ud, f, rc, p=None):
    if rc == 0:
        _ok.set()
        print('✅ paho-mqtt connected successfully → ' + HOST + ':' + str(PORT))
    else:
        print('❌ Connection failed, error code=' + str(rc))

c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id='workshop-test')
c.on_connect = _on_connect
try:
    c.connect(HOST, PORT, keepalive=5)
    c.loop_start()
    connected = _ok.wait(timeout=5)
    c.disconnect(); c.loop_stop()
    if not connected:
        print('⚠️ Timeout — re-run the "Start Mosquitto" cell above')
except Exception as e:
    print('❌ ' + str(e))
    print('  → Re-run the "Start Mosquitto" cell above first')


---
# Part 5 — Node-RED Pipeline

## Sensor data processing pipeline

```
[MQTT In: pump/sensors]
       │
[Parse & Validate]        ← validate schema, enrich with timestamp
       │
[Rolling Buffer 60]       ← keep last 60 readings in RAM (needs ≥ 5 to start)
       │
[Compute Trends & Stats]  ← avg_60, slope, rate_of_change, std_dev per sensor
       │                     overall_status = NORMAL / WARNING / CRITICAL
       │
       ├──────────────────────────────────────────────────────────────────►
       │   POST /simulate/inject (always)    ← enriched sensor_update → Dashboard WS
       │   payload includes: trends, overall_status, history (20pts for sparklines)
       │
       └─(anomaly_detected = true)──► [Throttle 60s] ──► POST /analyze
                                                              └─ Groq AI → ai_recommendation → Dashboard WS
```

**Why Node-RED adds real value here:**

| What Node-RED computes | What the dashboard uses it for |
|------------------------|--------------------------------|
| `trends[sensor].current` | Sensor gauge value (threshold-aware) |
| `trends[sensor].status` | Badge color (NORMAL/WARNING/CRITICAL) |
| `trends[sensor].trending` | Trend arrow (↑ DEGRADING / → STABLE) |
| `trends[sensor].avg_60` | Rolling average shown in sensor detail |
| `trends[sensor].history` | Sparkline (last 20 readings) |
| `overall_status` | Machine health banner & event log |

Node-RED runs **locally** on Colab. After the public URL is created (Part 6), you can view the pipeline visually at **NODERED_URL**.


### 📋 Practice: Upload `flows.json`

*Node-RED pipeline config — must be uploaded before starting Node-RED.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/nodered/`**
3. Drag and drop **`flows.json`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/nodered/flows.json` |
| 📍 Upload to | `/content/pump-iot-demo/nodered/flows.json` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/nodered/flows.json'
assert os.path.exists(_p), '❌ flows.json not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ flows.json ({_kb} KB) — OK')


> 💡 **flows.json** is pre-configured to work with local Mosquitto (`localhost:1883`)
> and FastAPI (`localhost:8000`). Node-RED will be started in **Part 5.2** below.


---
## 5.2 Start Node-RED & Deploy Flow

Node-RED runs on Colab. The flow is deployed **automatically via REST API** — no need to open a browser or click Deploy.

> 💡 **Why not use the UI?**
> When accessing Node-RED through a Cloudflare tunnel, the editor WebSocket times out → the Deploy button is greyed out.
> Using the API (`POST /flows`) is the standard way to deploy flows from a script — faster and no manual steps needed.


In [ ]:
import subprocess, time, os, requests, json as _json

NR_HOME = '/root/.node-red'
os.makedirs(NR_HOME, exist_ok=True)

flow_src = '/content/pump-iot-demo/nodered/flows.json'
if not os.path.exists(flow_src):
    print('❌ flows.json not found — upload it in step 5.1 first')
else:
    with open(flow_src, 'rb') as r, open(NR_HOME + '/flows.json', 'wb') as w:
        w.write(r.read())

    with open(NR_HOME + '/settings.js', 'w') as f:
        f.write('module.exports = {\n'
                '  uiPort: 1880,\n'
                '  httpAdminRoot: "/",\n'
                '  userDir: "/root/.node-red",\n'
                '  flowFile: "flows.json",\n'
                '  logging: { console: { level: "warn" } }\n'
                '};\n')

    # Capture stderr to display errors if Node-RED crashes
    nr_proc = subprocess.Popen(
        ['node-red', '--settings', NR_HOME + '/settings.js'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=NR_HOME
    )

    print('⏳ Waiting for Node-RED to start...')
    ready = False
    for i in range(45):
        time.sleep(2)
        if nr_proc.poll() is not None:
            out = nr_proc.stdout.read().decode(errors='replace')
            print('❌ Node-RED crashed. Output:')
            print(out[-2000:] if len(out) > 2000 else out)
            break
        try:
            if requests.get('http://localhost:1880', timeout=2).status_code < 500:
                ready = True
                break
        except: pass
        print(f'   {(i+1)*2}s...', end='\r')

    if ready:
        print(f'✅ Node-RED running — port 1880 ({(i+1)*2}s)')
        try:
            with open(flow_src) as f:
                flows = _json.load(f)
            payload = flows if isinstance(flows, list) else flows.get('flows', [])
            r = requests.post('http://localhost:1880/flows', json=payload,
                headers={'Content-Type': 'application/json'},
                timeout=10)
            if r.status_code in (200, 204):
                print('✅ Flow deployed (REST API)')
            else:
                print(f'⚠️ Deploy HTTP {r.status_code}: {r.text[:300]}')
        except Exception as e:
            print(f'⚠️ Deploy error: {e}')
    elif nr_proc.poll() is None:
        print('⚠️ Timed out (90s) — try re-running this cell')


---
# Part 6 — Dashboard & FastAPI Server

## Dashboard consists of 2 files

| File | Description |
|------|-------------|
| `index.html` | Main dashboard: 4 tabs (Live Feed · Timeline · AI Recommendation · Sensor Status) |
| `control.html` | Control Panel: simulate Normal / Warning / Critical scenarios |

The dashboard connects to the backend via **WebSocket** for real-time data:
```javascript
const ws = new WebSocket(`wss://${location.host}/ws`);
ws.onmessage = e => {
    const data = JSON.parse(e.data);
    updateCharts(data);     // Chart.js timeline
    updateGauges(data);     // SVG arc gauges
    updateSensorPage(data); // Sensor status grid
}
```


### 📋 Practice: Upload `index.html`

*Main dashboard with 4 tabs, real-time charts and SVG gauges.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/dashboard/`**
3. Drag and drop **`index.html`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/dashboard/index.html` |
| 📍 Upload to | `/content/pump-iot-demo/dashboard/index.html` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/index.html'
assert os.path.exists(_p), '❌ index.html not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ index.html ({_kb} KB) — OK')


### 📋 Practice: Upload `control.html`

*Control panel — used to simulate fault scenarios during the demo.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/dashboard/`**
3. Drag and drop **`control.html`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/dashboard/control.html` |
| 📍 Upload to | `/content/pump-iot-demo/dashboard/control.html` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/control.html'
assert os.path.exists(_p), '❌ control.html not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ control.html ({_kb} KB) — OK')


## 6.1 Start FastAPI server

After uploading `server.py` (Part 3) and the 2 dashboard files, start the backend:


In [ ]:
import subprocess, time
fastapi_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/pump-iot-demo/backend',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print('⏳ Waiting for FastAPI...')
for _ in range(20):
    time.sleep(1)
    line = fastapi_proc.stdout.readline().decode('utf-8', errors='replace')
    if 'Application startup complete' in line or 'Uvicorn running' in line:
        print(f'✅ FastAPI running (PID {fastapi_proc.pid}) | localhost:8000')
        break
else:
    if fastapi_proc.poll() is None:
        print('⚠️ Still starting... Run the health check cell below')
    else:
        print('❌ Exited early — check log:')
        print(fastapi_proc.stdout.read(2000).decode(errors='replace'))


In [ ]:
import requests, time
time.sleep(2)
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print('✅ Health check: ' + str(r.status_code))
    print('   ' + str(r.json()))
except Exception as e:
    print('❌ ' + str(e))

## 6.2 Create Public URL

Colab does not have a public IP. We use **ngrok** to expose FastAPI and Node-RED externally.

1. Sign up for free at **[ngrok.com](https://ngrok.com/)** → copy your authtoken
2. Fill in `NGROK_TOKEN` in the cell below
3. Run the cell — you will get a public URL for the dashboard and Node-RED editor

> ⏱️ URL is valid for the duration of your Colab session. Re-run this cell if the tunnel drops.


In [ ]:
import os, re, subprocess, time, requests

# ════════════════════════════════════════════════
# ✏️ Enter your ngrok authtoken
NGROK_TOKEN = ''   # get yours at: https://dashboard.ngrok.com/get-started/your-authtoken
# ════════════════════════════════════════════════

assert NGROK_TOKEN.strip(), '❌ Fill in NGROK_TOKEN above before running'

PUBLIC_URL  = None
NODERED_URL = None

def _update_env_and_restart(api_url):
    global PUBLIC_URL, fastapi_proc
    PUBLIC_URL = api_url
    env_path = '/content/pump-iot-demo/backend/.env'
    if os.path.exists(env_path):
        with open(env_path) as f: content = f.read()
        if 'PUBLIC_URL=' in content:
            content = re.sub(r'PUBLIC_URL=.*', 'PUBLIC_URL=' + PUBLIC_URL, content)
        else:
            content = content + '\nPUBLIC_URL=' + PUBLIC_URL
        with open(env_path, 'w') as f: f.write(content)
    if 'fastapi_proc' in dir() and fastapi_proc.poll() is None:
        fastapi_proc.terminate(); time.sleep(1)
    fastapi_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/pump-iot-demo/backend',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, 'PUBLIC_URL': PUBLIC_URL},
    )
    time.sleep(3)

print('📦 Installing pyngrok...')
import subprocess as _sp; _sp.run(['pip', 'install', '-q', 'pyngrok'])
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()

api_tunnel = ngrok.connect(8000, 'http')
nr_tunnel  = ngrok.connect(1880, 'http')
_update_env_and_restart(api_tunnel.public_url)
NODERED_URL = nr_tunnel.public_url

print()
print('=' * 65)
print('  🌐 DASHBOARD   : ' + PUBLIC_URL + '/dashboard/')
print('  🎮 CONTROLS    : ' + PUBLIC_URL + '/control/')
print('  📖 API DOCS    : ' + PUBLIC_URL + '/docs')
print('  ⚙️  NODE-RED    : ' + NODERED_URL)
print('=' * 65)
print('✅ ngrok tunnel created — FastAPI restarted with PUBLIC_URL')


---
# Part 7 — Sensor Simulator

## Data pipeline

```
sensor.csv ──► analyze_sensors.py ──► sensor_groups.json
                                            │
mqtt_replay.py ◄────────────────────────────┘
     │
     └──► publish pump/sensors ──► MQTT Broker ──► Node-RED ──► Dashboard
```

## Required files

| File | Location | Purpose |
|------|----------|---------|
| `sensor.csv` | `data/` | Raw dataset — 220K rows, 52 sensors |
| `analyze_sensors.py` | `scripts/` | Analyse CSV → generate `sensor_groups.json` |
| `sensor_groups.json` | `data/` | **Auto-generated** after running analyze |
| `mqtt_replay.py` | `scripts/` | Replay CSV over MQTT with time compression |

> ⚡ **Time compression**: 1 minute of real data = ~0.17s in demo (360× compression).
> Dataset of 220K rows ≈ 153 days — full replay completes in ~10 minutes.


### 📋 Practice: Upload `sensor.csv` & `analyze_sensors.py`

**File 1 — Raw dataset:**
1. Open the **Files panel** on the left
2. Navigate to: **`/content/pump-iot-demo/data/`**
3. Upload **`sensor.csv`**

**File 2 — Analysis script:**
1. Navigate to: **`/content/pump-iot-demo/scripts/`**
2. Upload **`analyze_sensors.py`**

| &nbsp; | Source (ZIP) | Upload to |
|--------|--------------|-----------|
| `sensor.csv` | `pump-iot-demo/data/sensor.csv` | `/content/pump-iot-demo/data/sensor.csv` |
| `analyze_sensors.py` | `pump-iot-demo/scripts/analyze_sensors.py` | `/content/pump-iot-demo/scripts/analyze_sensors.py` |

> After uploading both files, run the cell below to generate `sensor_groups.json`.


In [ ]:
import os, subprocess, sys

# Check that both required files exist
csv_path    = '/content/pump-iot-demo/data/sensor.csv'
script_path = '/content/pump-iot-demo/scripts/analyze_sensors.py'

for _p, _name in [(csv_path, 'sensor.csv'), (script_path, 'analyze_sensors.py')]:
    assert os.path.exists(_p), '❌ ' + _name + ' not found — please upload it again'
    _kb = max(1, os.path.getsize(_p) // 1024)
    print('✅ ' + _name + ' (' + str(_kb) + ' KB) — OK')

# Run analyze_sensors.py to generate sensor_groups.json
print()
print('⚙️  Running analyze_sensors.py...')
result = subprocess.run(
    [sys.executable, 'scripts/analyze_sensors.py',
     '--csv', 'data/sensor.csv',
     '--out', 'data/sensor_groups.json'],
    cwd='/content/pump-iot-demo',
    capture_output=True, text=True
)
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr[-500:])
else:
    grp_path = '/content/pump-iot-demo/data/sensor_groups.json'
    _kb = max(1, os.path.getsize(grp_path) // 1024)
    print('✅ sensor_groups.json (' + str(_kb) + ' KB) generated!')


### 📋 Practice: Upload `mqtt_replay.py`

*Sensor replay script — requires `sensor_groups.json` from the step above.*

1. Open the **Files panel** on the left
2. Navigate to: **`/content/pump-iot-demo/scripts/`**
3. Upload **`mqtt_replay.py`**

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/scripts/mqtt_replay.py` |
| 📍 Upload to | `/content/pump-iot-demo/scripts/mqtt_replay.py` |

## 7.1 Start the Simulator

After uploading, run the cell below. Open the **Dashboard URL** to see real-time data.

> 💡 The simulator starts from **row 0 (NORMAL state)** and replays the full dataset.
> Use **Part 10** to jump directly to the fault scenario during the demo.


In [ ]:
import subprocess, time, os, sys
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

replay_path = '/content/pump-iot-demo/scripts/mqtt_replay.py'
if not os.path.exists(replay_path):
    print('❌ mqtt_replay.py not found — please upload it following the instructions above')
elif not os.path.exists('/content/pump-iot-demo/data/sensor_groups.json'):
    print('❌ sensor_groups.json not found — run the analyze cell in Part 7 first')
else:
    sim_proc = subprocess.Popen(
        [sys.executable, 'scripts/mqtt_replay.py',
         '--csv',       'data/sensor.csv',
         '--config',    'data/sensor_groups.json',
         '--row-start', '0'],  # Start from row 0 — NORMAL state
        cwd='/content/pump-iot-demo',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print('🚀 Simulator running — NORMAL state (starting from row 0)')
    print('   Dashboard will receive data in ~5s')
    print()
    for _ in range(15):
        line = sim_proc.stdout.readline().decode('utf-8', errors='replace')
        if not line: break
        print(line.rstrip())
    print()
    print('💡 Open Dashboard: ' + (PUBLIC_URL + '/dashboard/' if 'PUBLIC_URL' in dir() and PUBLIC_URL else '(run cell 6.2 first)'))


---
# Part 8 — End-to-End Pipeline Check

Full pipeline is now running:
```
[mqtt_replay.py]
  └─ MQTT pub → [Mosquitto :1883] → [Node-RED :1880]
                                          │
                        ┌─────────────────┤
                        │                 │
               POST /simulate/inject  POST /analyze (on anomaly, throttled 60s)
               (enriched trends)          │
                        │          [Groq AI → ai_recommendation]
                        └─────────────────┘
                   [FastAPI :8000]
                        │ WebSocket
                   [Dashboard Browser]
```

Run the cell below to check each layer:


In [ ]:
import requests, time
print('=== Pipeline Status ===')
time.sleep(1)
for name, url in [('FastAPI /health', 'http://localhost:8000/health'),
                  ('Latest sensor',   'http://localhost:8000/latest')]:
    try:
        r = requests.get(url, timeout=3)
        d = r.json()
        if name == 'FastAPI /health':
            mqtt_ok  = d.get('mqtt_connected', False)
            has_data = d.get('has_data', False)
            last_st  = d.get('last_status', 'N/A')
            broker   = d.get('mqtt_broker', 'N/A')
            ws       = d.get('ws_clients', 0)
            print('  ✅ FastAPI running')
            print('     MQTT broker   : ' + broker)
            print('     MQTT connected: ' + ('✅ yes' if mqtt_ok else '❌ NO — check .env credentials'))
            print('     Has live data : ' + ('✅ yes (status=' + str(last_st) + ')' if has_data else '⚠️  not yet — start mqtt_replay.py'))
            print('     WS clients    : ' + str(ws))
        else:
            if d.get('status') == 'no_data':
                print('  ⚠️  /latest: No data yet — start the simulator (Part 7)')
            else:
                print('  ✅ /latest: status=' + str(d.get('machine_status')) +
                      ' health=' + str(round(d.get('health_score', 0), 1)))
    except Exception as e:
        print('  ❌ ' + name + ': ' + str(e))

if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print()
    print('🌐 ' + str(PUBLIC_URL) + '/dashboard/')


---
# Part 9 — AI Analysis & Email Alerts

## Analysis flow

```
Node-RED detects anomaly (overall_status = WARNING or CRITICAL)
    └─ POST /analyze  (throttled: max 1 call per 60s)
           └─ Groq AI  →  JSON response
                  └─ broadcast ai_recommendation  →  Dashboard WS  →  AI tab

Dashboard detects WARNING/CRITICAL from sensor_update
    └─ POST /alert  (cooldown: 120s for CRITICAL, 300s for WARNING)
           └─ Resend API  →  HTML email  →  Inbox
```

> 💡 AI analysis and email alerts are **independent**: Node-RED triggers the AI, the dashboard triggers the email.
> This means emails are sent even if the AI tab is not open, and the AI panel updates even if email is not configured.


In [ ]:
import requests, json

_base = str(PUBLIC_URL) if 'PUBLIC_URL' in dir() and PUBLIC_URL else 'http://localhost:8000'

# ── Test 1: AI analysis via /analyze ──────────────────────────────────────
print('📤 Test 1: AI analysis (POST /analyze)...')
snapshot = {
    'timestamp':      __import__('datetime').datetime.utcnow().isoformat(),
    'machine_status': 'BROKEN',
    'health_score':   22.5,
    'overall_status': 'CRITICAL',
    'sensors': {
        'vibration':   {'current': 7.82, 'avg_60_readings': 5.4, 'trend': 'DEGRADING', 'status': 'CRITICAL', 'rate_of_change': 0.12},
        'temperature': {'current': 96.5, 'avg_60_readings': 82.1, 'trend': 'DEGRADING', 'status': 'CRITICAL', 'rate_of_change': 0.8},
        'pressure':    {'current': 9.8,  'avg_60_readings': 8.1,  'trend': 'STABLE',    'status': 'WARNING',  'rate_of_change': 0.05},
        'flow_rate':   {'current': 85.0, 'avg_60_readings': 140.0,'trend': 'DEGRADING', 'status': 'CRITICAL', 'rate_of_change': -2.1},
    }
}
try:
    r = requests.post(_base + '/analyze', json={'snapshot': snapshot}, timeout=30)
    if r.status_code == 200:
        ai = r.json()
        print('  ✅ AI responded!')
        print('     Risk level  :', ai.get('risk_level'))
        print('     Confidence  :', ai.get('confidence'))
        print('     Summary     :', ai.get('summary', '')[:80])
        print('     Hours to fail:', ai.get('estimated_hours_to_failure'))
    else:
        print(f'  ❌ HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print('  ❌ ' + str(e))

print()

# ── Test 2: Email alert via /alert ────────────────────────────────────────
print('📧 Test 2: Email alert (POST /alert)...')
alert_body = {
    'level':        'CRITICAL',
    'health_score': 22.5,
    'message':      'Test alert — vibration and temperature critical.',
    'sensor_summary': {
        'Vibration':   '7.82 mm/s',
        'Temperature': '96.5 °C',
        'Pressure':    '9.8 bar',
        'Flow Rate':   '85.0 m³/h',
    },
    'sensor_statuses': {
        'Vibration':   'CRITICAL',
        'Temperature': 'CRITICAL',
        'Pressure':    'WARNING',
        'Flow Rate':   'CRITICAL',
    },
    'ai_risk_level':              'CRITICAL',
    'estimated_hours_to_failure': 6,
    'estimated_savings':          513500,
}
try:
    r = requests.post(_base + '/alert', json=alert_body, timeout=10)
    result = r.json()
    status = result.get('status', '?')
    if status == 'sent':
        print('  ✅ Email sent → ' + str(result.get('to')))
    elif status == 'skipped':
        print('  ℹ️  Skipped: ' + result.get('reason', ''))
        print('     (Set RESEND_API_KEY and ALERT_TO in .env to enable emails)')
    else:
        print('  ❌ Error: ' + str(result))
except Exception as e:
    print('  ❌ ' + str(e))


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')
key = os.environ.get('RESEND_API_KEY', '')
if key and len(key) > 10:
    print('✅ Resend configured')
    print('   From : ' + os.environ.get('ALERT_FROM',''))
    print('   To   : ' + os.environ.get('ALERT_TO',''))
    print('   Emails sent on WARNING/CRITICAL (60s cooldown)')
else:
    print('ℹ️  Resend not configured — emails will be skipped')
    print('   To enable: sign up at resend.com → set RESEND_API_KEY in .env')


---
# Part 10 — Demo: Simulate a Fault

| Scenario | Symptoms | Real-world cause |
|----------|----------|-----------------|
| 🔴 Bearing Wear | Vibration rises, temperature increases | Worn bearing surfaces |
| 🔴 Cavitation | Pressure fluctuates, flow drops | Low inlet pressure |
| 🔴 Overheating | Temperature spikes rapidly | Insufficient lubrication |


In [ ]:
import subprocess, time, sys, os

print('🔴 Switching to CRITICAL mode...')
if 'sim_proc' in dir() and sim_proc.poll() is None:
    sim_proc.terminate()
    time.sleep(0.5)

sim_proc = subprocess.Popen(
    [sys.executable, 'scripts/mqtt_replay.py',
     '--csv',              'data/sensor.csv',
     '--config',           'data/sensor_groups.json',
     '--start-at-anomaly',
     '--anomaly-offset',   '0',
     '--quiet'],
    cwd='/content/pump-iot-demo',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print('✅ Simulator restarted — jumping directly to the BROKEN row')
print('   CRITICAL data will appear on the Dashboard in ~3s')
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('   🌐 ' + str(PUBLIC_URL) + '/dashboard/')


In [ ]:
import subprocess, sys, os

if 'sim_proc' in dir() and sim_proc.poll() is None:
    sim_proc.terminate()

sim_proc = subprocess.Popen(
    [sys.executable, 'scripts/mqtt_replay.py',
     '--csv',       'data/sensor.csv',
     '--config',    'data/sensor_groups.json',
     '--row-start', '0', '--quiet'],
    cwd='/content/pump-iot-demo',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print('✅ Reset to NORMAL mode — replaying from the start of the CSV')


---
# Part 11 — Monitoring & Troubleshooting


In [ ]:
import requests
def svc(name, proc, url=None):
    alive = proc is not None and proc.poll() is None
    icon = '🟢' if alive else '🔴'
    out = icon + ' ' + name
    if alive and url:
        try:
            r = requests.get(url, timeout=2)
            out += ' | HTTP ' + str(r.status_code)
        except: out += ' | unreachable'
    print(out)

print('═'*46)
svc('FastAPI :8000',   locals().get('fastapi_proc'), 'http://localhost:8000/health')
svc('Simulator',       locals().get('sim_proc'))
for n,v in [('Mosquitto','mosquitto_proc'),('Node-RED','nr_proc')]:
    if v in dir(): svc(n, eval(v))
print('═'*46)
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('\n🌐 ' + PUBLIC_URL + '/dashboard/')

| Symptom | Cause | Fix |
|---------|-------|-----|
| Dashboard blank | FastAPI not running | Re-run Part 6.1 |
| No data received | Node-RED not running | Re-run Part 5.2 |
| AI tab spinning | No API key configured | Add GROQ_API_KEY → restart backend |
| Email not sent | RESEND_KEY invalid | Check resend.com Logs |
| MQTT error | Mosquitto not started | Re-run Part 4.1 |
| ngrok URL expired | Session timeout | Re-run Part 6.2 |
| AI returns [MOCK] + 429 | Quota exceeded | Auto-falls back to llama-3.1-8b-instant |


---
# Part 12 — Wrap-up & Exercises

## 🎉 System complete!

| What you built | Production equivalent |
|---------------|----------------------|
| Mosquitto MQTT | AWS IoT Core / Azure IoT Hub |
| Node-RED | Industrial Edge Gateway |
| FastAPI + WebSocket | Real-time telemetry platform |
| AI analysis | Predictive maintenance ML |
| Resend email | PagerDuty / OpsGenie |

---

## 🚀 Extension exercises

**Beginner:**
1. Add an `acoustic_emission` sensor type (ultrasonic sound)
2. Adjust alert thresholds to match real pump datasheet values
3. Add a second language to the AI system prompt

**Intermediate:**
4. Save history to SQLite + add `/history?hours=24` endpoint
5. Add Basic Auth to the Control Panel
6. Integrate Grafana to visualise data from SQLite

**Advanced:**
7. Replace rule-based detection with **Isolation Forest** (scikit-learn)
8. Attach a real MPU-6050 sensor to a Raspberry Pi
9. Migrate to **AWS IoT Core + Lambda + DynamoDB**

---

**Docs:** [Mosquitto](https://mosquitto.org/documentation/) · [Node-RED](https://nodered.org/docs/) · [FastAPI](https://fastapi.tiangolo.com/)
